# Lab Week 10 — Hugging Face Tutorial Lab: `pipeline()`

In this lab, we will explore what can ```pipeline()``` do, and what behind ```pipeline()```

Note:
If you use a newer version of the Hugging Face Transformers library, such as Transformers 5.8.1, you may have access to more models and additional pipeline tasks, such as:

* `image-text-to-text`
* `text-to-audio`

However, in the latest version, the `summarization` and `question-answering` pipelines may no longer be available by default.

## 0. Setup

In [1]:
#!pip install "transformers==4.38.2"
!pip install torch datasets evaluate transformers accelerate
#!pip install -U "transformers>=4.46" datasets evaluate "accelerate>=1.1.0"

import transformers
import accelerate

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

#import sys
#print(sys.executable)

transformers: 5.12.1
accelerate: 1.14.0


## 1. Transformers, what can they do?

In [2]:
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

Torch: 2.12.0+cu130
Transformers: 5.12.1


## Sentiment-Analysis
We didn't assign a specific model for the task, 
So, the default model is ``` distilbert/distilbert-base-uncased-finetuned-sst-2-english ```

In [3]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

# Test on multiple sentences
texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!"
]

result2 = classifier(texts)
print(result2)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9982948899269104}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


## Zero-Shot Classification

The default model is ``` facebook/bart-large-mnli ```

MNLI: Multi-Genre Natural Language Inference

Zero-shot classification does not magically know every possible label. Instead, Hugging Face converts the classification problem into a natural language inference problem. For each candidate label, it creates a hypothesis such as “This text is about sports.” Then an MNLI model checks whether the original text entails that hypothesis. facebook/bart-large-mnli is used by default because it is a BART model fine-tuned on the MNLI dataset, so it is already good at judging entailment, contradiction, and neutrality.

In [4]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445952534675598, 0.1119769737124443, 0.04342775419354439]}

## Text Generation
Previous defaulted model is: ```openai-community/gpt2 ```

Recent revised default model is  ```HuggingFaceTB/SmolLM3-3B```

In [7]:
generator = pipeline("text-generation", model="gpt2")
generator("In this course, I will teach you how to") #slow - 3B 


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, I will teach you how to build a self-contained world in a simple to understand way.'}]

In [ ]:
#Assign more parameters to control the generation
generator = pipeline("text-generation", model="distilgpt2")

outputs = generator(
    "In this course, I will teach you how to",
    max_new_tokens=30,
    truncation=True,
    num_return_sequences=2
)

for i, output in enumerate(outputs, 1):
    print(f"Output {i}:")
    print(output["generated_text"])
    print()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output 1:
In this course, I will teach you how to do a little bit of math in the next couple of weeks. I’m going to cover the basics, but I’m going to

Output 2:
In this course, I will teach you how to make your own life easier. It is not an entire course and I will teach you how to make yourself better.



There are many



## Fill Mask

Default model is ``` distilbert/distilroberta-base```

In [9]:
unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.19619834423065186,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052722081542015,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

## NER: Named Entity Recognition
Aggregate token-level predictions into word/entity-level predictions using a simple grouping rule.


In [10]:
ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Peiyuan and I work at Conestoga College in Waterloo.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': 0.9846198,
  'word': 'Peiyuan',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': 0.97662514,
  'word': 'Conestoga College',
  'start': 33,
  'end': 50},
 {'entity_group': 'LOC',
  'score': 0.9702344,
  'word': 'Waterloo',
  'start': 54,
  'end': 62}]

## Image-text-to-text 
```image-text-to-text``` is a multimodal task which is one of vision-language models.

You need to provide both an image and a text instruction, such as “What is shown in this image?”. The model will then generate a description of the image. 

We use ```HuggingFaceTB/SmolVLM-256M-Instruct``` which is smaller and easier for a demonstration.

The default model is 

![a bee on a flower](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg)

In [13]:
from transformers import pipeline

pipe = pipeline(
    "image-text-to-text",
    model="HuggingFaceTB/SmolVLM-256M-Instruct"
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
            },
            {
                "type": "text",
                "text": "What is shown in this image?"
            }
        ]
    }
]

result = pipe(
    text=messages,
    max_new_tokens=1000
)

assistant_response = result[0]["generated_text"][-1]["content"]
print(assistant_response)


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 The image features a close-up view of a flower with a distinct pink hue. The flower is prominently featured in the foreground, with its petals fully open and a yellow center. The petals are delicate and delicate, with a slight sheen that suggests they are freshly bloomed. The flower is surrounded by green foliage, which provides a natural backdrop to the main subject.

In the center of the image, there is an insect. The insect is small and appears to be in the process of feeding on the flower. The insect has a dark body and a light-colored head, which is typical of insects that feed on nectar. The insect's proboscis is extended, and it is likely that it is drinking nectar from the flower.

The background of the image is slightly blurred, which helps to focus the viewer's attention on the flower and the insect. The background consists of a variety of colors, including green, brown, and red, which add to the natural and vibrant appearance of the image.

The overall composition of the im

## * Question-Answering
Default model: ```distilbert/distilbert-base-cased-distilled-squad```
Only works for lower version of Transformer, you can install Transformer==4.38.2

In [9]:
from transformers import pipeline

question_answerer = pipeline("question-answering")
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


{'score': 0.6949764490127563, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}

## * Summarization
Default model: ```sshleifer/distilbart-cnn-12-6```

Only works for lower version of Transformer, you can install Transformer==4.38.2

In [10]:
summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""
)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'summary_text': ' America has changed dramatically during recent years . The number of engineering graduates in the U.S. has declined in traditional engineering disciplines such as mechanical, civil,    electrical, chemical, and aeronautical engineering . Rapidly developing economies such as China and India continue to encourage and advance the teaching of engineering .'}]

## 2. What behind `pipeline()`? 
### The simplest usage: `pipeline("sentiment-analysis")`

The Hugging Face `pipeline()` API hides three steps:

```text
1. Preprocessing: raw text → tokens → token IDs → tensors
2. Model inference: tensors → Transformer → classification head → logits
3. Postprocessing: logits → softmax probabilities → labels and scores
```

The default sentiment-analysis checkpoint used in the Hugging Face tutorial is:

```python
distilbert-base-uncased-finetuned-sst-2-english
```
A **checkpoint** saved pretrained/fine-tuned model.

In [14]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
# Test bias in the model: "Muted" may often be associated with negative sentiment

texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!",
    "The movie was okay, but not very exciting.",
    "The movie has a muted visual style.",
]

results = classifier(texts)

for text, result in zip(texts, results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I've been waiting for a Hugging Face course my whole life.
Result: {'label': 'POSITIVE', 'score': 0.9982948899269104}
--------------------------------------------------------------------------------
Text: I hate this so much!
Result: {'label': 'NEGATIVE', 'score': 0.9994558691978455}
--------------------------------------------------------------------------------
Text: The movie was okay, but not very exciting.
Result: {'label': 'NEGATIVE', 'score': 0.9978721141815186}
--------------------------------------------------------------------------------
Text: The movie has a muted visual style.
Result: {'label': 'NEGATIVE', 'score': 0.9978736639022827}
--------------------------------------------------------------------------------


### Step 1: Preprocessing - Tokenization
The default model is ```distilbert/distilbert-base-uncased-finetuned-sst-2-english```
You can choose a spedific model

In [15]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
print(checkpoint)

distilbert-base-uncased-finetuned-sst-2-english


In [16]:
# [CLS]=101(classification); [SEP]=102(end of sentence); [PAD]=0; [UNK]=100; [MASK]=103
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_inputs = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!",
]

inputs = tokenizer(
    raw_inputs,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

#### What can Tokenizer do? 
The tokenizer converts the text into a format that a Transformer model can understand. A tokenizer usually outputs two important things:

1. `input_ids` are token IDs. For example, a word or subword is mapped to an integer.
2. `attention_mask` tells the model which positions are real tokens and which positions are padding.
    Usually:
    ```text
    1 = real token
    0 = padding token
    ```


#### Notice special tokens
For BERT-style models:
- `[CLS]` is often used as the sentence-level representation.
- `[SEP]` marks the end of a sentence.
- `[PAD]` is added to make sequences in the same batch have the same length.

Play with tokenizer in: https://huggingface.co/spaces/Xenova/the-tokenizer-playground

In [18]:
print(inputs)
print("input_ids shape:", inputs["input_ids"].shape)
print("attention_mask shape:", inputs["attention_mask"].shape)

print("\ninput_ids:")
print(inputs["input_ids"])

print("\nattention_mask:")
print(inputs["attention_mask"])

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662,  2227,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}
input_ids shape: torch.Size([2, 16])
attention_mask shape: torch.Size([2, 16])

input_ids:
tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662,  2227,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 

In [19]:
for i, sentence_ids in enumerate(inputs["input_ids"]):
    tokens = tokenizer.convert_ids_to_tokens(sentence_ids)
    print(f"Sentence {i + 1}:")
    print(tokens)
    print()

Sentence 1:
['[CLS]', 'i', "'", 've', 'been', 'waiting', 'for', 'a', 'hugging', 'face', 'course', 'my', 'whole', 'life', '.', '[SEP]']

Sentence 2:
['[CLS]', 'i', 'hate', 'this', 'so', 'much', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']



### Step 2: Model inference
### *2.1 Pass inputs through the base model*

First, we use `AutoModel`.
`AutoModel` loads the base Transformer model **without the classification head**.
It outputs hidden states/features, not final sentiment labels.


In [20]:
from transformers import AutoModel

base_model = AutoModel.from_pretrained(checkpoint)
base_outputs = base_model(**inputs)

print(base_outputs.last_hidden_state.shape)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 16, 768])


#### Understanding the shape

The output shape is usually:  [batch_size, sequence_length, hidden_size]

For this example, [2, 16, 768]:
- `2`: two input sentences
- `16`: token length after padding
- `768`: hidden vector dimension for each token

This is not yet a classification result.

### *2.2 Use a model with a classification head*

```AutoModel``` does choose the correct base model architecture from the checkpoint, but it only loads the base Transformer body. 

If you want sentiment classification, you need ```AutoModelForSequenceClassification ```, which adds a task-specific classification head on top of the base Transformer.

In [21]:
from transformers import AutoModelForSequenceClassification

classification_model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
outputs = classification_model(**inputs)

print(outputs.logits)
print("logits shape:", outputs.logits.shape)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[-3.1071,  3.2654],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)
logits shape: torch.Size([2, 2])


#### What are the outputs?
`logits` are the raw, unnormalized scores produced by the final layer of the model. They are **not probabilities**.

For sentiment analysis with two labels, each sentence gets two logits: ```[score_for_NEGATIVE, score_for_POSITIVE]```
    So the shape is: [batch_size, number_of_labels], 
        e.g.[2, 2]: There are 2 sentences into a 2-class sentiment model.

**output**: {[-3.1071,  3.2654],[4.1692, -3.3464]} 
1. in the first sentence,"I've been waiting for a Hugging Face course my whole life.", negative=-3.1071, positive=3.2654
2. and the second sentence, "I hate this so much!", negative=4.1692, positive =-3.3464

In [22]:
print("id2label mapping:")
print(classification_model.config.id2label)

print("\nlabel2id mapping:")
print(classification_model.config.label2id)

id2label mapping:
{0: 'NEGATIVE', 1: 'POSITIVE'}

label2id mapping:
{'NEGATIVE': 0, 'POSITIVE': 1}


### Step 3: Post Analysis - Convert logits to probabilities with softmax

The model output logits must be converted into probabilities.
We use:

```python
softmax(logits)
```
The largest logit usually becomes the largest probability.


In [23]:
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(probabilities)


tensor([[1.7051e-03, 9.9829e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


### Convert probabilities into labels
Now we manually reproduce the final output of the pipeline.

In [24]:
predicted_class_ids = torch.argmax(probabilities, dim=-1)

for i, text in enumerate(raw_inputs):
    class_id = predicted_class_ids[i].item()
    label = classification_model.config.id2label[class_id]
    score = probabilities[i][class_id].item()

    print("Text:", text)
    print("Predicted label:", label)
    print("Score:", score)
    print("-" * 80)


Text: I've been waiting for a Hugging Face course my whole life.
Predicted label: POSITIVE
Score: 0.9982948899269104
--------------------------------------------------------------------------------
Text: I hate this so much!
Predicted label: NEGATIVE
Score: 0.9994558691978455
--------------------------------------------------------------------------------


### Compare manual result with `pipeline()`

Now we check whether our manual process gives the same meaning as the high-level pipeline.


In [25]:
classifier = pipeline("sentiment-analysis", model=checkpoint, tokenizer=checkpoint)

pipeline_results = classifier(raw_inputs)

for text, result in zip(raw_inputs, pipeline_results):
    print("Text:", text)
    print("Pipeline result:", result)
    print("-" * 80)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I've been waiting for a Hugging Face course my whole life.
Pipeline result: {'label': 'POSITIVE', 'score': 0.9982948899269104}
--------------------------------------------------------------------------------
Text: I hate this so much!
Pipeline result: {'label': 'NEGATIVE', 'score': 0.9994558691978455}
--------------------------------------------------------------------------------


In [26]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

# Test on multiple sentences
texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I hate this so much!"
]

result = classifier(texts)
print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9982948899269104}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


### Try your own examples
Can you modify the following texts and observe the output.

In [27]:
student_texts = [
    "This course is difficult but very useful.",
    "The homework is confusing and frustrating.",
    "I am not sure whether I like this movie.",
    "The product is not bad at all.",
]

student_results = classifier(student_texts)

for text, result in zip(student_texts, student_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)


Text: This course is difficult but very useful.
Result: {'label': 'POSITIVE', 'score': 0.9954234957695007}
--------------------------------------------------------------------------------
Text: The homework is confusing and frustrating.
Result: {'label': 'NEGATIVE', 'score': 0.9991816878318787}
--------------------------------------------------------------------------------
Text: I am not sure whether I like this movie.
Result: {'label': 'NEGATIVE', 'score': 0.9955669045448303}
--------------------------------------------------------------------------------
Text: The product is not bad at all.
Result: {'label': 'POSITIVE', 'score': 0.998784601688385}
--------------------------------------------------------------------------------


### Limitation: sentence meaning can be subtle

A sentiment model may struggle with:

- sarcasm
- mixed opinions
- negation
- domain-specific language
- long context

For example:

```text
This movie is so good that I almost fell asleep.
```

The surface words may look positive, but the actual meaning is negative/sarcastic.


In [28]:
tricky_texts = [
    "This movie is so good that I almost fell asleep.",
    "The phone is cheap, but surprisingly reliable.",
    "I expected to hate it, but I actually loved it.",
    "Not bad at all.",
]

tricky_results = classifier(tricky_texts)

for text, result in zip(tricky_texts, tricky_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)


Text: This movie is so good that I almost fell asleep.
Result: {'label': 'POSITIVE', 'score': 0.9998125433921814}
--------------------------------------------------------------------------------
Text: The phone is cheap, but surprisingly reliable.
Result: {'label': 'POSITIVE', 'score': 0.9980798959732056}
--------------------------------------------------------------------------------
Text: I expected to hate it, but I actually loved it.
Result: {'label': 'POSITIVE', 'score': 0.9998137354850769}
--------------------------------------------------------------------------------
Text: Not bad at all.
Result: {'label': 'POSITIVE', 'score': 0.99928218126297}
--------------------------------------------------------------------------------
